In [1]:
# type: ignore

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


load_dotenv()

model = ChatOpenAI(
    model=os.environ.get("MODEL_NAME"),
    temperature=0,
    base_url=os.environ.get("COMPATIBLE_BASE_URL"),
    api_key=os.environ.get("COMPATIBLE_API_KEY"),
)

# 1. Define state

In [2]:
import operator
from typing import Annotated, Literal, TypedDict


class AgentInput(TypedDict):
    """Simple input state for each subagent."""

    query: str


class AgentOutput(TypedDict):
    """Output from each subagent."""

    source: str
    result: str


class Classification(TypedDict):
    """A single routing decision: which agent to call with what query."""

    source: Literal["github", "notion", "slack"]
    query: str


class RouterState(TypedDict):
    query: str
    classifications: list[Classification]
    results: Annotated[
        list[AgentOutput], operator.add
    ]  # Reducer collects parallel results
    final_answer: str

# 2. Define tools for each vertical

In [3]:
from langchain.tools import tool


@tool
def search_code(query: str, repo: str = "main") -> str:
    """Search code in GitHub repositories."""
    return f"Found code matching '{query}' in {repo}: authentication middleware in src/auth.py"


@tool
def search_issues(query: str) -> str:
    """Search GitHub issues and pull requests."""
    return f"Found 3 issues matching '{query}': #142 (API auth docs), #89 (OAuth flow), #203 (token refresh)"


@tool
def search_prs(query: str) -> str:
    """Search pull requests for implementation details."""
    return "PR #156 added JWT authentication, PR #178 updated OAuth scopes"


@tool
def search_notion(query: str) -> str:
    """Search Notion workspace for documentation."""
    return "Found documentation: 'API Authentication Guide' - covers OAuth2 flow, API keys, and JWT tokens"


@tool
def get_page(page_id: str) -> str:
    """Get a specific Notion page by ID."""
    return "Page content: Step-by-step authentication setup instructions"


@tool
def search_slack(query: str) -> str:
    """Search Slack messages and threads."""
    return "Found discussion in #engineering: 'Use Bearer tokens for API auth, see docs for refresh flow'"


@tool
def get_thread(thread_id: str) -> str:
    """Get a specific Slack thread."""
    return "Thread discusses best practices for API key rotation"

# 3. Create specialized agents

In [4]:
from langchain.agents import create_agent


github_agent = create_agent(
    model,
    tools=[search_code, search_issues, search_prs],
    system_prompt=(
        "You are a GitHub expert. Answer questions about code, "
        "API references, and implementation details by searching "
        "repositories, issues, and pull requests."
    ),
)

notion_agent = create_agent(
    model,
    tools=[search_notion, get_page],
    system_prompt=(
        "You are a Notion expert. Answer questions about internal "
        "processes, policies, and team documentation by searching "
        "the organization's Notion workspace."
    ),
)

slack_agent = create_agent(
    model,
    tools=[search_slack, get_thread],
    system_prompt=(
        "You are a Slack expert. Answer questions by searching "
        "relevant threads and discussions where team members have "
        "shared knowledge and solutions."
    ),
)

# 4. Build the router workflow

In [6]:
from pydantic import BaseModel, Field
from langgraph.types import Send
from langchain.messages import SystemMessage, HumanMessage


router_llm = model


# Define structured output schema for the classifier
class ClassificationResult(BaseModel):
    """Result of classifying a user query into agent-specific sub-questions."""

    classifications: list[Classification] = Field(
        description="List of agents to invoke with their targeted sub-questions"
    )


def classify_query(state: RouterState) -> dict:
    """Classify query and determine which agents to invoke."""
    structured_llm = router_llm.with_structured_output(ClassificationResult)

    result = structured_llm.invoke(
        [
            SystemMessage("""Analyze this query and determine which knowledge bases to consult.
                          For each relevant source, generate a targeted sub-question optimized for that source.
                          
                          Available sources:
                          - github: Code, API references, implementation details, issues, pull requests
                          - notion: Internal documentation, processes, policies, team wikis
                          - slack: Team discussions, informal knowledge sharing, recent conversations
                          
                          Return ONLY the sources that are relevant to the query. Each source should have
                          a targeted sub-question optimized for that specific knowledge domain.
                          Output the sources and targeted sub-question in JSON format.
                          
                          Example for "How do I authenticate API requests?":
                          - github: "What authentication code exists? Search for auth middleware, JWT handling"
                          - notion: "What authentication documentation exists? Look for API auth guides"
                          (slack omitted because it's not relevant for this technical question)"""),
            HumanMessage(state["query"]),
        ]
    )

    return {"classifications": result.classifications}  # type: ignore


def route_to_agents(state: RouterState) -> list[Send]:
    """Fan out to agents based on classifications."""
    return [Send(c["source"], {"query": c["query"]}) for c in state["classifications"]]


def query_github(state: AgentInput) -> dict:
    """Query the GitHub agent."""
    result = github_agent.invoke({"messages": [HumanMessage(state["query"])]})
    return {"results": [{"source": "github", "result": result["messages"][-1].content}]}


def query_notion(state: AgentInput) -> dict:
    """Query the Notion agent."""
    result = notion_agent.invoke({"messages": [HumanMessage(state["query"])]})
    return {"results": [{"source": "notion", "result": result["messages"][-1].content}]}


def query_slack(state: AgentInput) -> dict:
    """Query the Slack agent."""
    result = slack_agent.invoke({"messages": [HumanMessage(state["query"])]})
    return {"results": [{"source": "slack", "result": result["messages"][-1].content}]}


def synthesize_results(state: RouterState) -> dict:
    """Combine results from all agents into a coherent answer."""
    if not state["results"]:
        return {"final_answer": "No results found from any knowledge source."}

    # Format results for synthesis
    formatted = [
        f"**From {r['source'].title()}:**\n{r['result']}" for r in state["results"]
    ]

    synthesis_response = router_llm.invoke(
        [
            SystemMessage(f"""Synthesize these search results to answer the original question: "{state["query"]}"
                          
                          - Combine information from multiple sources without redundancy
                          - Highlight the most relevant and actionable information
                          - Note any discrepancies between sources
                          - Keep the response concise and well-organized"""),
            HumanMessage("\n\n".join(formatted)),
        ]
    )

    return {"final_answer": synthesis_response.content}

# 5. Compile the workflow

In [7]:
from langgraph.graph import StateGraph, START, END


workflow = (
    StateGraph(RouterState)
    .add_node("classify", classify_query)
    .add_node("github", query_github)
    .add_node("notion", query_notion)
    .add_node("slack", query_slack)
    .add_node("synthesize", synthesize_results)
    .add_edge(START, "classify")
    .add_conditional_edges("classify", route_to_agents, ["github", "notion", "slack"])
    .add_edge("github", "synthesize")
    .add_edge("notion", "synthesize")
    .add_edge("slack", "synthesize")
    .add_edge("synthesize", END)
    .compile()
)

print(workflow.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify(classify)
	github(github)
	notion(notion)
	slack(slack)
	synthesize(synthesize)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify;
	classify -.-> github;
	classify -.-> notion;
	classify -.-> slack;
	github --> synthesize;
	notion --> synthesize;
	slack --> synthesize;
	synthesize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



# 6. TEST

In [8]:
result = workflow.invoke({"query": "怎么对API请求进行认证？"})  # type: ignore

print("Original query:", result["query"])
print("\nClassifications:")
for c in result["classifications"]:
    print(f"  {c['source']}: {c['query']}")
print("\n" + "=" * 60 + "\n")
print("Final Answer:")
print(result["final_answer"])

Original query: 怎么对API请求进行认证？

Classifications:
  github: How do I authenticate API requests?
  notion: What authentication documentation exists? Look for API auth guides


Final Answer:
要对 API 请求进行认证，可以采用以下几种方式，具体取决于你的系统设计和安全需求：

### 1. **使用中间件进行认证（来自 GitHub）**
- 项目中已实现了一个认证中间件 `src/auth.py`，用于验证传入请求的凭据。
- 建议查看该文件以了解具体的认证逻辑和集成方式。

### 2. **支持的认证机制（来自 Notion 文档）**
Notion 中的 **"API Authentication Guide"** 提到了三种主流认证方式：
- **OAuth2 流程**：适用于第三方应用授权，支持授权码、客户端凭证等模式。
- **API Keys**：适合服务间通信或简单客户端认证，通常通过请求头（如 `X-API-Key`）传递。
- **JWT（JSON Web Tokens）**：常用于基于令牌的无状态认证，包含用户信息和签名，可设置过期时间。

### 3. **补充信息（来自 GitHub Issues）**
- **Issue #142**：提供了 API 认证的文档说明。
- **Issue #89**：详细描述了 OAuth 的实现流程。
- **Issue #203**：解释了如何刷新过期的认证令牌（如 JWT 或 OAuth access token）。

---

### 建议操作
- 如果你正在开发或调试 API，首先查阅 `src/auth.py` 的实现。
- 若需选择认证方案，可根据场景决定：
  - 内部服务调用 → **API Key**
  - 用户登录态管理 → **JWT**
  - 第三方集成 → **OAuth2**

如需更详细的步骤或代码示例，我可以帮你提取 Notion 页面内容或分析 `auth.py` 的逻辑。


# 7. Add memory

In [10]:
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_knowledge_base(query: str) -> str:
    """Search across multiple knowledge sources (GitHub, Notion, Slack).

    Use this to find information about code, documentation, or team discussions.
    """
    result = workflow.invoke({"query": query})  # type: ignore
    return result["final_answer"]


conversational_agent = create_agent(
    model,
    tools=[search_knowledge_base],
    system_prompt=(
        "You are a helpful assistant that answers questions about our organization. "
        "Use the search_knowledge_base tool to find information across our code, "
        "documentation, and team discussions."
    ),
    checkpointer=InMemorySaver(),
)

In [11]:
from langchain_core.runnables import RunnableConfig


config = RunnableConfig({"configurable": {"thread_id": "user-123"}})

result = conversational_agent.invoke(
    {"messages": [HumanMessage("怎么对API请求进行认证？")]},
    config,
)
print(result["messages"][-1].content)

result = conversational_agent.invoke(
    {"messages": [HumanMessage("那 rate limit呢？")]},
    config,
)
print(result["messages"][-1].content)

我们的API认证系统主要通过 `src/auth.py` 中的中间件实现，包含以下几种方式：

## 主要认证方式

1. **JWT认证** - 基于JSON Web Tokens的令牌认证
   - 在PR #156中引入
   - 适用于大多数API端点

2. **OAuth集成** - 支持第三方应用授权
   - 在PR #178中增强了作用域控制
   - Issue #89详细讨论了OAuth授权流程

3. **令牌刷新机制** - 处理过期令牌的自动刷新
   - 在Issue #203中实现

## 实施建议

- 查看 `src/auth.py` 文件了解具体的实现逻辑
- 根据你的使用场景选择合适的认证方式：
  - 内部服务调用：推荐JWT
  - 第三方应用集成：使用OAuth

需要注意的是，目前的API认证文档还不够完善（Issue #142），如果你在实施过程中遇到问题，我可以帮你查找更具体的技术细节。

你希望了解哪种认证方式的具体实现？
关于API限流（rate limiting），我在现有的知识库中没有找到明确的实现细节。虽然我们有完善的认证系统（JWT/OAuth），但限流机制似乎还没有被明确记录或讨论。

## 当前状态分析

**代码库中未发现明确的限流实现**：
- 没有相关的GitHub Issues或PR专门讨论限流
- 认证相关的PR中也没有提及限流功能

**可能的情况**：
1. 限流功能尚未实现
2. 限流在基础设施层（如Nginx、API网关）处理，不在应用代码中
3. 限流实现存在但未被文档化

## 建议的排查步骤

我可以帮你进一步搜索：

1. **检查代码库**中是否有以下限流相关的关键字：
   - `rate_limit`、`throttle`、`limiter`
   - 相关库如 `slowapi`、`redis`（用于计数）
   - 配置文件中的限流设置

2. **查看部署配置**，确认是否在反向代理层处理限流

你希望我优先搜索哪个方向？或者你能告诉我你的具体需求场景吗？比如你是遇到了限流问题，还是想要了解如何实施限流？
